# QwerySmith v3.0 — Eval Matrix (serving + eval + report)

Serves the four-system matrix and runs `eval` + `report` on identical questions and identical frozen packs. **Runtime → L4 GPU** (runs the 30B-AWQ row and can also serve the 8B rows; a T4 works for rows 1–2 only).

| Cell | What it does |
|---|---|
| 1 | Setup + artifact download from Drive |
| 2 | Serve row 1+2 (8B base + LoRA hot-swap) — vLLM `
| 3 | Serve row 3 (30B-A3B AWQ) — vLLM, ~17GB |
| 4 | Frontier key + eval run |
| 5 | Report + failure folders |
| 6 | Zip results to Drive |

In [ ]:
# ===== Cell 1: setup =====
%pip install -q uv vllm

import torch
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> L4 GPU'
vram = torch.cuda.get_device_properties(0).total_memory // 2**20
print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {vram} MB')

!git clone -q https://github.com/Cyrax321/QwerySmith-1.0.git /content/qwerysmith || (cd /content/qwerysmith && git pull -q)
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/qwerysmith'
DATA = '/content/drive/MyDrive/qwerysmith_v3'

# restore trained adapters + prepared artifacts from the training run
import zipfile, glob, os, shutil
zips = sorted(glob.glob(f'{DATA}/run_train_*.zip'))
assert zips, 'no training run zip in Drive — run t4_train.ipynb first'
with zipfile.ZipFile(zips[-1]) as z:
    z.extractall(REPO)
print('restored:', zips[-1])

# raw data for ingest (identical DB state as training)
os.makedirs(f'{REPO}/datasets/olist/raw', exist_ok=True)
for f in os.listdir(f'{DATA}/raw'):
    if f.endswith('.csv'):
        shutil.copy(f'{DATA}/raw/{f}', f'{REPO}/datasets/olist/raw/{f}')

# rebuild DB + packs (idempotent; packs hash-verified by the registry)
%cd {REPO}
!uv run python -m qwery_smith ingest olist
!uv run python -m qwery_smith validate olist
!uv run python -m qwery_smith retrieve olist

In [ ]:
# ===== Cell 2: serve rows 1+2 — Qwen3-8B base with LoRA hot-swap =====
# run in a terminal (or via subprocess): vLLM serves base + both adapters;
# rows 1/2 then share weights/process — the plan §5.2 requirement
import glob, pathlib, subprocess
run_dir = sorted(pathlib.Path(f'{REPO}/runs/olist').glob('train_*'))[-1]
adapters = sorted((run_dir / 'adapters').glob('adapter_seed*'))
print('adapters found:', [a.name for a in adapters])
assert len(adapters) == 3, 'need 3 seed adapters'

lora_flags = ' '.join(
    [f"--enable-lora --lora-modules ft={{name}}={a}" for name, a in
     [(f'ft-s{i+1}', a) for i, a in enumerate(adapters)]]
)
# one adapter is served at a time under the stable name 'ft' — the systems.yaml
# model_id stays constant across seed passes, so eval needs no config edits
SEED_UNDER_TEST = 1   # candidate row = median-EX seed chosen after the first pass

cmd = (
    f"vllm serve Qwen/Qwen3-8B --port 8000 --enable-lora "
    f"--lora-modules ft={run_dir}/adapters/adapter_seed{SEED_UNDER_TEST} "
    f"--max-lora-rank 16 --gpu-memory-utilization 0.90 --max-model-len 8192"
)
print('start this in a terminal:')
print(cmd)

In [ ]:
# ===== Cell 3: serve row 3 — Qwen3-30B-A3B-Instruct-2507 AWQ =====
# ~17GB weights -> L4 24GB. Run in a second terminal (or a second Colab
# instance; eval tolerates servers on different hosts via systems.yaml):
print(
    "vllm serve Qwen/Qwen3-30B-A3B-Instruct-2507-AWQ "
    "--port 8001 --max-model-len 8192 --gpu-memory-utilization 0.92"
)

In [ ]:
# ===== Cell 4: frontier key + run eval =====
# frontier: pinned model version recorded in the run manifest; Olist is
# public data, so API use is sanctioned (plan §5.2)
import os
os.environ['FRONTIER_API_KEY'] = input('frontier API key: ').strip()

# point systems.yaml endpoints at this instance (or edit the file once):
import yaml
sysyaml_path = f'{REPO}/datasets/olist/systems.yaml'
sysyaml = yaml.safe_load(open(sysyaml_path))
for s in sysyaml['systems']:
    if s['name'] == 'qwen3-8b-base':
        s['base_url'] = 'http://localhost:8000/v1'; s['model_id'] = 'Qwen/Qwen3-8B'
    if s['name'] == 'qwen3-8b-ft':
        s['base_url'] = 'http://localhost:8000/v1'; s['model_id'] = 'ft'
    if s['name'] == 'qwen3-30b-a3b':
        s['base_url'] = 'http://localhost:8001/v1'
    if s['name'] == 'frontier':
        s['base_url'] = os.environ.get('FRONTIER_BASE_URL', 'https://api.example.com/v1')
        s['model_id'] = os.environ.get('FRONTIER_MODEL', '<pinned-frontier-model>')
open(sysyaml_path, 'w').write(yaml.safe_dump(sysyaml, sort_keys=False))

# the matrix: identical questions, identical packs, all four rows
!uv run python -m qwery_smith eval olist
# repeat the candidate row for seeds 2 and 3: restart the Cell 2 server
# with SEED_UNDER_TEST=2 (same 'ft' name), then re-run eval — each pass
# writes its own run dir; `report` reads the latest, or merge manually
# per the 3-seed aggregation in plan §7.1 (mean EX ± range).

In [ ]:
# ===== Cell 5: report + failure folders =====
!uv run python -m qwery_smith report olist

import pathlib
run = sorted(pathlib.Path(f'{REPO}/runs/olist').glob('*'))[-1]
print(pathlib.Path(run / 'report.md').read_text())

In [ ]:
# ===== Cell 6: zip results to Drive =====
!cd {REPO} && zip -qr /content/drive/MyDrive/qwerysmith_v3/eval_results.zip runs/olist datasets/olist/systems.yaml
print('results zipped; report.md + failure folders + raw outputs included')